In [1]:
import pickle
import dgl
import ast
import argparse
from sklearn import preprocessing
from sklearn.preprocessing import MinMaxScaler
from model import *
from utils import *
import numpy as np
import pandas as pd
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from dgl.nn.pytorch import GATConv
import logging 
import datetime
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn import linear_model
from sklearn import metrics
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score,f1_score
import os

/Users/mengping/anaconda3/envs/py37/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = torch.device('cpu')
graph_path = './data/'
label_path = './data/'
feature_path = './data/'


In [4]:
#gene
gene_meta_paths = [['gg'],['gm','mg']]
gene_graphs,_ = dgl.data.utils.load_graphs(graph_path+'/gene/gene_graph.bin')
gene_g = gene_graphs[0]
gene_g = gene_g.to(device)
print (gene_g)

#mirna
mirna_meta_paths = [['mm'],['mg','gm']]
mirna_graphs,_ = dgl.data.utils.load_graphs(graph_path+'/miRNA/mirna_graph.bin')
mirna_g = mirna_graphs[0]
mirna_g = mirna_g.to(device)

Graph(num_nodes={'gene': 12944, 'mirna': 867},
      num_edges={('gene', 'gg', 'gene'): 627420, ('gene', 'gm', 'mirna'): 261025, ('mirna', 'mg', 'gene'): 261025},
      metagraph=[('gene', 'gene', 'gg'), ('gene', 'mirna', 'gm'), ('mirna', 'gene', 'mg')])


In [5]:
g=[gene_g,mirna_g]

In [6]:
gene_test_mask = pd.read_csv(label_path+'/gene/gene_test_mask_fold_0.txt',sep='\t',header=None)
gene_test_label_mask = pd.read_csv(label_path+'/gene/gene_test_fold_0.txt',sep='\t',header=None)
mirna_test_mask = pd.read_csv(label_path+'/miRNA/mirna_test_mask_fold_0.txt',sep='\t',header=None)
mirna_test_label_mask = pd.read_csv(label_path+'/miRNA/mirna_test_fold_0.txt',sep='\t',header=None)

gene_train_mask = pd.read_csv(label_path+'/gene/gene_trainval_mask_fold_0.txt',sep='\t',header=None)
gene_train_label_mask = pd.read_csv(label_path+'/gene/gene_trainval_fold_0.txt',sep='\t',header=None)
mirna_train_mask = pd.read_csv(label_path+'/miRNA/mirna_trainval_mask_fold_0.txt',sep='\t',header=None)
mirna_train_label_mask = pd.read_csv(label_path+'/miRNA/mirna_trainval_fold_0.txt',sep='\t',header=None)


In [7]:
gene_test_mask_fold = torch.tensor(gene_test_mask.iloc[:,0].to_numpy()).to(device)
gene_test_label_mask_fold = torch.tensor(gene_test_label_mask.iloc[:,0].to_numpy()).to(device)
gene_train_mask_fold = torch.tensor(gene_train_mask.iloc[:,0].to_numpy()).to(device)
gene_train_label_mask_fold = torch.tensor(gene_train_label_mask.iloc[:,0].to_numpy()).to(device)
mirna_test_mask_fold = torch.tensor(mirna_test_mask.iloc[:,0].to_numpy()).to(device)
mirna_test_label_mask_fold = torch.tensor(mirna_test_label_mask.iloc[:,0].to_numpy()).to(device)
mirna_train_mask_fold = torch.tensor(mirna_train_mask.iloc[:,0].to_numpy()).to(device)
mirna_train_label_mask_fold = torch.tensor(mirna_train_label_mask.iloc[:,0].to_numpy()).to(device)

In [8]:
gene_train_label_mask

,0
0,1
1,1
2,1
3,0
4,0
...,...
2215,0
2216,0
2217,0
2218,0


In [9]:
train_index=[gene_train_mask_fold,mirna_train_mask_fold]
train_label=[gene_train_label_mask_fold,mirna_train_label_mask_fold]
test_index=[gene_test_mask_fold,mirna_test_mask_fold]
test_label=[gene_test_label_mask_fold,mirna_test_label_mask_fold]
all_meta_paths=[[['gg'],['gm','mg']],
                [['mm'],['mg','gm']]]

In [10]:
gene_fea=pd.read_csv(feature_path+'gene/total_feature_500.csv',sep=',',header=None)
mirna_fea=pd.read_csv(feature_path+'miRNA/6.feautre_miRNA_index_total.csv',sep=',',header=None)

In [17]:
gene_fea

,0,1,2,3,4,5,6,7,8,9,...,54,55,56,57,58,59,60,61,62,63
0,0.000000,0.000000,0.000000,0.045474,0.062031,0.004340,0.000000,0.003597,0.057510,0.000000,...,0.102967,0.009384,-0.059198,0.140384,0.078644,0.196181,-0.089560,0.341877,0.007439,0.111045
1,0.101469,0.114481,0.053236,0.068058,0.670553,0.031057,0.118088,0.000000,0.572198,0.053475,...,-0.628506,-0.418915,-0.311997,-0.311055,0.221890,-0.358416,0.329412,-0.397676,0.503337,-0.099926
2,0.000000,0.020391,0.000000,0.045358,0.051372,0.021684,0.019502,0.000000,0.030644,0.000000,...,0.198408,-0.691813,-0.434057,-0.026059,0.087662,-0.444373,0.352443,-0.549445,0.376260,-1.001897
3,0.012829,0.010627,0.000000,0.022648,0.000000,0.011000,0.000000,0.000000,0.030737,0.000000,...,-0.359450,-0.432546,-0.673727,-0.338856,0.712949,-0.798008,-0.708609,-0.245409,0.514037,-0.010694
4,0.000000,0.010459,0.028161,0.000000,0.048383,0.016111,0.000000,0.000000,0.000000,0.000000,...,-5.392841,-3.209208,1.490494,-5.555310,8.014996,-1.593752,8.287197,-2.246793,-8.143787,3.861393
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12939,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.020542,0.000000,0.000000,0.000000,...,0.168850,0.280642,0.234262,0.284598,-0.240188,0.227030,-0.253851,0.339618,0.083119,0.106095
12940,0.000000,0.010612,0.028318,0.022767,0.025903,0.005496,0.000000,0.000000,0.000000,0.000000,...,-0.132870,0.493204,0.157843,0.369651,-0.198677,0.196211,0.179648,0.176243,0.014653,-0.035454
12941,0.000000,0.005309,0.000000,0.022772,0.000000,0.000000,0.020454,0.000000,0.030760,0.000000,...,0.571895,0.719392,-0.021886,0.899764,-0.833604,0.090304,-0.866547,0.304842,0.894069,-0.701551
12942,0.025493,0.025121,0.028160,0.000000,0.011565,0.021188,0.079436,0.003593,0.029840,0.026908,...,0.007914,0.296897,0.296884,0.131838,-0.235913,0.292774,-0.132656,0.293518,0.004340,0.097711


In [11]:
def proprecess_train_test(data):
    minmax_scaler=preprocessing.MinMaxScaler()
    data_train_scaler=minmax_scaler.fit_transform(data)
    data_train_scaler_df=pd.DataFrame(data_train_scaler)
    data_train_scaler_df.columns=data.columns
    data_train_scaler_df.index=data.index
    return (data_train_scaler_df)
    
mirna_pro_fea=proprecess_train_test(mirna_fea)
gene_fea_tensor=torch.tensor(gene_fea.values)
mirna_fea_tensor=torch.tensor(mirna_pro_fea.values)
gene_fea_tensor=gene_fea_tensor.to(torch.float32)
mirna_fea_tensor=mirna_fea_tensor.to(torch.float32)
    
gene_fea_tensor=gene_fea_tensor.to(device)
mirna_fea_tensor=mirna_fea_tensor.to(device)
node_features=[gene_fea_tensor,mirna_fea_tensor]

In [12]:
auc = [];aupr = [];acc = [];micro = [];macro = []    
    
target_dim=64
in_size=[64,64]
hidden_size=[256,256]
out_size=[2,2]
num_heads=[4]
dropout=0.2
lr=0.003
weight_decay=0.0005

In [13]:
model = GMHAN(target_dim=target_dim,
        all_meta_paths=all_meta_paths,
                    in_size=in_size,
                    hidden_size=hidden_size,
                    out_size=out_size,
                    num_heads=num_heads,
                    dropout=dropout).to(device)
    
stopper = EarlyStopping(100)

In [14]:
loss_fcn = torch.nn.CrossEntropyLoss()
optim = torch.optim.Adam(lr=lr, weight_decay=weight_decay, params=model.parameters())
    
dataset_train_index=train_index
dataset_train_label=train_label
dataset_test_index=test_index
dataset_test_label=test_label  

In [15]:
def main_test(model,g,node_features,test_index,test_label):
    model.eval()
    with torch.no_grad():
        g_logits, m_logits, m_logits2, g_probs, m_probs, m_probs2,g_fea, m_fea = model(g,node_features,  dataset_test_index)
        loss_g = loss_fcn(g_logits, dataset_test_label[0].reshape(-1))
        loss_m = loss_fcn(m_logits, dataset_test_label[1].reshape(-1))
        loss = loss_g+loss_m
            
        #gene
        g_pred1 = g_logits.cpu().numpy()
        g_label = dataset_test_label[0].cpu().numpy()
        g_acc = accuracy_score(g_label, np.argmax(g_pred1, axis=1))
        g_auc = roc_auc_score(g_label, g_probs[:, 1].cpu().numpy())
        g_aupr = average_precision_score(g_label, g_probs[:, 1].cpu().numpy())
        #mirna
        m_pred1 = m_logits.cpu().numpy()
        m_label = dataset_test_label[1].cpu().numpy()
        m_acc = accuracy_score(m_label, np.argmax(m_pred1, axis=1))
        m_auc = roc_auc_score(m_label, m_probs[:, 1].cpu().numpy())
        m_aupr = average_precision_score(m_label, m_probs[:, 1].cpu().numpy())
    return loss,g_acc,g_auc,g_aupr,m_acc,m_auc,m_aupr,g_probs,m_probs,m_probs2

In [16]:
gene_best_auc=0
gene_best_aupr=0
gene_best_acc=0
mirna_best_auc=0
mirna_best_aupr=0
mirna_best_acc=0
best_epoch=0
g_best_prob=[]
m_best_prob=[]
m_best_prob2=[]
    
for epoch in range(2):
    model.train()
    g_logits, m_logits, m_logits2, g_probs, m_probs, m_probs2, g_fea, m_fea = model(g,node_features,dataset_train_index)
    
    loss_g = loss_fcn(g_logits, dataset_train_label[0].reshape(-1))
    loss_m = loss_fcn(m_logits, dataset_train_label[1].reshape(-1))
    loss = loss_g+loss_m
    optim.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optim.step()
    loss,g_acc,g_auc,g_aupr,m_acc,m_auc,m_aupr,g_probs,m_probs,m_probs2=main_test(model,g,node_features,  dataset_test_index,dataset_test_label)
    
    if g_auc>gene_best_auc:
        gene_best_auc = g_auc
        gene_best_aupr = g_aupr
        gene_best_acc = g_acc
        mirna_best_auc = m_auc
        mirna_best_aupr = m_aupr
        mirna_best_acc = m_acc
        best_epoch = epoch
        g_best_prob=g_probs
        m_best_prob=m_probs
        m_best_prob2=m_probs2
print ("---------")
print (loss, best_epoch, gene_best_auc,gene_best_aupr,gene_best_acc,mirna_best_auc,mirna_best_aupr,mirna_best_acc)

---------
tensor(1.2551) 1 0.38371480781738776 0.23135152443687235 0.7009009009009008 0.7312348668280872 0.5068840215828434 0.6781609195402298
